In [ ]:

def print_puzzle(puzzle_rect):
    for i in puzzle_rect:
        for j in i:
            if(j == 0):
                print("■", end="")
            elif(j == 2):
                print("X", end=" ")
            elif(j == 3):
                print("·", end="")
            else:
                print(" ", end=" ")
            # print(end=" ")
        print()

def isOnBorder(pos, n):
    if(pos[0] <= 0 or pos[0] >= n - 1 or pos[1] <= 0 or pos[1] >= n - 1):
        return True 
    else:
        return False

def getNeighbor(pos, n, i):
    neighbor = []
    for i in [(i, 0), (-1 * i, 0), (0, i), (0, -1 * i)]:
        if (not isOnBorder((pos[0] + i[0], pos[1] + i[1]), n)):
            neighbor.append((pos[0] + i[0], pos[1] + i[1]))
    return neighbor



In [276]:
import random as r
def puzzle_init(start, end, n):
    puzzle_rect = [[0 for i in range(n)] for j in range(n)]

    if(start == end):
        print("起点和终点不能相同")
        return puzzle_rect
    elif(isOnBorder(start, n) or isOnBorder(end, n)):
        print("起点和终点不能在边界上")
        return puzzle_rect
    else:
        wall = []
        puzzle_rect[start[0]][start[1]] = 1
        for neighbor in getNeighbor(start, n, 2):
            wall.append((neighbor, start))
        while wall:
            index = r.randint(0, len(wall)-1)
            cur_pos , from_pos = wall.pop(index)
            if (puzzle_rect[cur_pos[0]][cur_pos[1]] == 0):
                mid_pos = ((from_pos[0] + cur_pos[0])//2, (from_pos[1] + cur_pos[1])//2)
                puzzle_rect[mid_pos[0]][mid_pos[1]] = 1
                puzzle_rect[cur_pos[0]][cur_pos[1]] = 1
                for i in getNeighbor(cur_pos, n, 2):
                    if (puzzle_rect[i[0]][i[1]] == 0):
                        wall.append((i, cur_pos))

        puzzle_rect[start[0]][start[1]] = 2
        puzzle_rect[end[0]][end[1]] = 2
        return puzzle_rect


In [233]:
import io
from contextlib import redirect_stdout
from IPython.display import display, HTML

def update_maze_display(puzzle_rect, handle=None):
    """
    用 print_puzzle 的输出刷新 Jupyter 单元格显示（无闪烁）。
    首次调用不传 handle，后续传入返回的 handle 即可。
    """
    # 捕获 print_puzzle 的输出
    with io.StringIO() as buf, redirect_stdout(buf):
        print_puzzle(puzzle_rect)          # 调用你已有的打印函数
        output = buf.getvalue()
    
    html = '<pre>' + output.rstrip() + '</pre>'
    if handle is None:
        handle = display(HTML(html), display_id=True)
    else:
        handle.update(HTML(html))
    return handle

In [234]:
import math as m
import heapq
import time

class Node:
    def __init__(self, pos, g, h, parent=None):
        self.pos = pos
        self.g = g
        self.h = h
        self.f = g + h
        self.parent = parent
    def __eq__(self, other):
        if isinstance(other, Node):
            return self.pos == other.pos
        return False
    def __hash__(self):
        return self.pos

def astar(puzzle_rect, start, end, n):
    handle = display(HTML(''), display_id=True)

    def get_distance(pos, end):
        return (abs(pos[0] - end[0]) + abs(pos[1] - end[1]))
    
    open_list = []
    close_list = []

    g = 0
    counter = 0
    h = get_distance(start, end)
    start_node = Node(start, g, h)
    heapq.heappush(open_list, (start_node.f, counter, start_node))
    counter += 1

    handle = None
    while open_list:
        cur_f, _, cur_node = heapq.heappop(open_list)

        if (cur_node.pos == end):
            for i in range(len(puzzle_rect)):
                for j in range(len(puzzle_rect)):
                    if(puzzle_rect[i][j] == 3):
                        puzzle_rect[i][j] = 1
            while cur_node.parent:
                cur_node = cur_node.parent
                if(cur_node.pos == start):
                    continue
                puzzle_rect[cur_node.pos[0]][cur_node.pos[1]] = 3
            
            update_maze_display(puzzle_rect, handle)
            print("已找到唯一通路")
            print("迷宫起点:",start)
            print("迷宫终点:",end)
            return

        handle = update_maze_display(puzzle_rect, handle)
        time.sleep(0.1)
        neighbor_list = getNeighbor(cur_node.pos, n, 1)

        for neighbor in neighbor_list:
            if (not neighbor in close_list and puzzle_rect[neighbor[0]][neighbor[1]] != 0):
                g = cur_node.g + 1
                h = get_distance(cur_node.pos, end)
                neighbor_node = Node(neighbor, g, h, cur_node)
                heapq.heappush(open_list, (neighbor_node.f, counter, neighbor_node))
                counter += 1
        
        if (puzzle_rect[cur_node.pos[0]][cur_node.pos[1]] == 1):
            puzzle_rect[cur_node.pos[0]][cur_node.pos[1]] = 3
        close_list.append(cur_node.pos)
        

            


In [278]:
import time

n = int(input("迷宫阶数"))
n = n-1

start_time = time.time
start = (1, 1)
end = (n-2, n-2)

puzzle_rect = puzzle_init(start, end, n)
astar(puzzle_rect, start, end, n)
end_time = time.time

# print(f"消耗时间:{(end_time-start_time):.4f}秒")

已找到唯一通路
迷宫起点: (1, 1)
迷宫终点: (47, 47)
